In [ ]:
valida esse:

from pyspark.sql.functions import *
from pyspark.sql.functions import min, max, datediff, col, when

from pyspark.sql.types import DateType
%md
#1. Seleção de público
%md
**Conceito:** todos os cartões gerados
df_publico = spark.sql("""SELECT num_conta,
                                num_cpf_cnpj_cliente,
                                dat_abertura_conta,
                                cod_logo,
                                cod_canal_venda,
                                cod_campanha_venda,
                                vlr_limite_credito_inicial

                            FROM gold.dim_cartao_credito_conta
                                --where dat_abertura_conta > '2023-01-01' and dat_abertura_conta < --'2023-09-01' >> puxa a base inteira para já ter safras recentes
                        """)
#manter só uma conta por CPF: a mais recente, com maior limite inicial (se tiver mais de um, remove aleatoriamente)
#para tirar o efeito de migração de logo

agrupado = df_publico.groupBy("num_cpf_cnpj_cliente").agg(max("dat_abertura_conta").alias("dat_abertura_conta"),max("vlr_limite_credito_inicial").alias("vlr_limite_credito_inicial"))

df_publico2 = df_publico.join(agrupado, ["num_cpf_cnpj_cliente", "dat_abertura_conta","vlr_limite_credito_inicial"], "inner")

df_publico2 = df_publico2.dropDuplicates(["num_cpf_cnpj_cliente"])
#tamanho da base
df_publico2.count()

#verifica se só tem cpf
df_publico2.select(min("num_cpf_cnpj_cliente"), max("num_cpf_cnpj_cliente")).show()
%md
**análise descritiva**
df_publico_desc = df_publico2.toPandas()
df_publico_desc['dat_abertura_conta'] = pd.to_datetime(df_publico_desc['dat_abertura_conta'], format='%Y-%m-%d')
volumetria = df_publico_desc.groupby('dat_abertura_conta')['num_conta'].count().reset_index()
plt.figure(figsize=(30, 8)) #tamanho da figura
plt.bar(volumetria['dat_abertura_conta'],volumetria['num_conta'],color='green') #tipo de gráfico e seleção de dados
plt.title('Volumetria de abertura de conta de cartão, por dia') #título
plt.xlabel('Data de abertura da conta') #eixo X
plt.ylabel('Volumetria') #eixo Y
plt.xticks(rotation=90, va='top') #rotação do valores do eixo X
plt.grid(True) #linhas de grades ao fundo
plt.show()
#-- volumetria por dia da semana
df_publico_desc['dat_abertura_conta_weekday'] = df_publico_desc['dat_abertura_conta'].dt.strftime('%A')
df_publico_desc['dat_abertura_conta_weekday_n'] = df_publico_desc['dat_abertura_conta'].dt.weekday

volumetria_weekday = df_publico_desc.groupby(['dat_abertura_conta_weekday','dat_abertura_conta_weekday_n'])['num_conta'].count().reset_index()
volumetria_weekday = volumetria_weekday.sort_values('dat_abertura_conta_weekday_n')  #usa o número do dia da semana para ordenar, se não ele ordena alfabético no gráfico

#-- volumetria por mês
df_publico_desc['dat_abertura_conta_month'] = df_publico_desc['dat_abertura_conta'].dt.strftime('%Y-%m')
volumetria_month = df_publico_desc.groupby('dat_abertura_conta_month')['num_conta'].count().reset_index()

plt.figure(figsize=(15, 10))

plt.subplot(2, 1, 1)
plt.bar(volumetria_weekday['dat_abertura_conta_weekday'],volumetria_weekday['num_conta'],color='black')
plt.title('Volumetria de abertura de conta de cartão, por dia da semana', fontsize=20)
plt.xlabel('Dia da semana da abertura da conta'+'\n'+' ')
plt.ylabel('Volumetria')
plt.xticks(rotation=90, va='top')
plt.grid(False)
plt.text(7.0, 0.35, 'praticamente sem abertura'+'\n'+'no final de semana',color='blue')

plt.subplot(2, 1, 2)
plt.bar(volumetria_month['dat_abertura_conta_month'],volumetria_month['num_conta'],color='black')
plt.title('Volumetria de abertura de conta de cartão, por mês', fontsize=20)
plt.xlabel('Mês da abertura da conta'+'\n'+' ')
plt.ylabel('Volumetria')
plt.xticks(rotation=-90, va='top')
plt.grid(False)
plt.text(6.0, 0.35, 'volumetria estável',color='blue')

plt.tight_layout()
plt.show()
%md
#2. Target
%md
**Conceito:** considerado como ativado (1) se realizou ao menos uma compra nos 30 dias posteriores à data de abertura
df_publico_spark = spark.createDataFrame(df_publico2)
df_publico_spark.createOrReplaceTempView("df_publico_spark")

df_publico_target = spark.sql("""  SELECT a.*, b.dat_primeira_compra, b.flg_tipo_compra
                                FROM df_publico AS a LEFT JOIN gold.fato_cartao_credito_ativacao AS b
                                ON a.num_conta = b.num_conta
""")

df_publico_target.count()
#Python (pandas)
#df_publico_target = df_publico_target.toPandas()
#df_publico_target['dat_primeira_compra'] = pd.to_datetime(df_publico_target['dat_primeira_compra'], format='%Y-%m-%d')
#df_publico_target['dias_ativacao'] = df_publico_target['dat_primeira_compra'] - df_publico_target['dat_abertura_conta']
#df_publico_target['dias_ativacao'] = df_publico_target['dias_ativacao'].dt.days
#df_publico_target['target'] = (df_publico_target['dias_ativacao'] <=60).astype(int)

#Pyspark

# Calcula a diferença de dias entre a primeira compra e a abertura da conta
df_publico_target = df_publico_target.withColumn("dias_ativacao", datediff(col("dat_primeira_compra"), col("dat_abertura_conta")))

# Cria a coluna target com base na condição: ativado se comprou em até 60 dias
df_publico_target = df_publico_target.withColumn("target", when(col("dias_ativacao") <= 60, 1).otherwise(0))
%md
**Análise descritiva**
df_publico_target_desc = df_publico_target.toPandas()
graph_target = pd.crosstab(df_publico_target_desc['dat_abertura_conta'],df_publico_target_desc['target']).reset_index()
graph_target['perc_target'] = graph_target[1]/(graph_target[1]+graph_target[0])
plt.figure(figsize=(30, 8)) #tamanho da figura
plt.plot(graph_target['dat_abertura_conta'],graph_target['perc_target'],color='green') #tipo de gráfico e seleção de dados
plt.title('taxa de ativação, por dia', fontsize=20) #título
plt.xlabel('Data de abertura da conta') #eixo X
plt.ylabel('taxa de ativação') #eixo Y
plt.xticks(rotation=90, va='top') #rotação do valores do eixo X
plt.grid(True) #linhas de grades ao fundo
plt.show()
graph_target_month = pd.crosstab(df_publico_target_desc['dat_abertura_conta_month'],df_publico_target_desc['target']).reset_index()
graph_target_month['perc_target'] = graph_target_month[1]/(graph_target_month[1]+graph_target_month[0])

graph_target_weekday = pd.crosstab([df_publico_target_desc['dat_abertura_conta_weekday_n'],df_publico_target['dat_abertura_conta_weekday']],df_publico_target_desc['target']).reset_index()
graph_target_weekday['perc_target'] = graph_target_weekday[1]/(graph_target_weekday[1]+graph_target_weekday[0])
plt.figure(figsize=(15, 10))

plt.subplot(2, 1, 1)
plt.plot(graph_target_month['dat_abertura_conta_month'],graph_target_month['perc_target'],color='black') #tipo de gráfico e seleção de dados
plt.title('taxa de ativação, por mês', fontsize=20) #título
plt.xlabel('Mês de abertura da conta') #eixo X
plt.ylabel('taxa de ativação') #eixo Y
plt.ylim(bottom=0, top=0.5)
plt.xticks(rotation=90, va='top') #rotação do valores do eixo X
plt.grid(False) #linhas de grades ao fundo

plt.subplot(2, 1, 2)
plot_target_weekday = plt.plot(graph_target_weekday['dat_abertura_conta_weekday'],graph_target_weekday['perc_target']) #tipo de gráfico e seleção de dados
plt.title('taxa de ativação, por dia da semana', fontsize=20) #título
plt.xlabel('Dia da semana da abertura da conta') #eixo X
plt.ylabel('taxa de ativação') #eixo Y
plt.ylim(bottom=0, top=0.5)
plt.xticks(rotation=90, va='top') #rotação do valores do eixo X
plt.grid(False) #linhas de grades ao fundo
plt.setp(plot_target_weekday,color='blue',linewidth = 3, linestyle = '--')
plt.tight_layout()
plt.show()
%md
#Salva base
df_publico_target_spark_f.write.mode("overwrite").saveAsTable("data_science.df_dummy_model_activation_dataprep")